In [2]:
import os
import re
import json
from typing import Dict, Any, List, Tuple, Sequence

import pandas as pd
import networkx as nx

# =========================
# Config
# =========================
DATA_PATH = "./training_data_normalized.csv"
GEXF_PATH = "./graph_GES.gexf"

OUT_BASE = "./add_feature_dataset"
ALG = "GES"

COL_PREFIX = "edge"

# ✅ weight 없이 만들려면 False
USE_EDGE_WEIGHT = False
STRICT_WEIGHT = True  # USE_EDGE_WEIGHT=True일 때만 의미 있음

# --- train/val/test split (shared) ---
TRAIN_RATIO = 0.7
VAL_RATIO = 0.1
TEST_RATIO = 0.2
RANDOM_STATE = 42
SHUFFLE_BEFORE_SPLIT = True
SAVE_SPLIT_INDICES = True

# --- metadata toggle ---
WRITE_METADATA = False

# --- output name prefix ---
DATA_PREFIX = "data_with_features"

# --- feature list csv ---
WRITE_FEATURE_LIST = True
FEATURE_LIST_FILENAME = "feature_list.csv"

# --- drop index-like from base/graph ---
DROP_INDEXLIKE_FROM_BASE = True
DROP_INDEXLIKE_FROM_GRAPH = True

TARGET_CANDIDATES = ["label", "target", "y", "failure", "bank_failure", "default", "is_failed"]


# =========================
# Helpers
# =========================
def is_int_str(x: str) -> bool:
    return re.fullmatch(r"-?\d+", str(x)) is not None


def is_indexlike_col_name(c: str) -> bool:
    cl = str(c).strip().lower()
    return cl.startswith("unnamed") or cl in {"index", "_index"} or cl.endswith("_index")


def drop_indexlike_columns(df: pd.DataFrame) -> pd.DataFrame:
    drop_cols = [c for c in df.columns if is_indexlike_col_name(c)]
    if drop_cols:
        print(f"[DROP] index-like columns: {drop_cols}")
        df = df.drop(columns=drop_cols)

    bad = [c for c in df.columns if str(c).strip().lower().startswith("unnamed")]
    if bad:
        raise RuntimeError(f"[FATAL] Unnamed columns still exist after drop: {bad}")
    return df


def detect_target_col(df: pd.DataFrame) -> str:
    for c in TARGET_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError(f"Target column not found among {TARGET_CANDIDATES}")


def ensure_digraph(G) -> nx.DiGraph:
    if isinstance(G, nx.MultiDiGraph):
        return G
    if isinstance(G, nx.DiGraph):
        return G
    return nx.DiGraph(G)


def clean_graph_nodes(G: nx.DiGraph, alg: str) -> nx.DiGraph:
    nodes = list(G.nodes())
    bad_nodes = [n for n in nodes if is_indexlike_col_name(str(n))]
    if bad_nodes:
        print(f"[GRAPH DROP:{alg}] removed index-like nodes -> {bad_nodes[:20]}" + (" ..." if len(bad_nodes) > 20 else ""))
        G = G.copy()
        G.remove_nodes_from(bad_nodes)
    return G


def build_node_to_col_mapping(df: pd.DataFrame, G: nx.DiGraph) -> Dict[Any, str]:
    cols = list(df.columns)
    nodes = list(G.nodes())

    # 1) exact match
    if all(str(n) in df.columns for n in nodes):
        return {n: str(n) for n in nodes}

    # 2) integer nodes -> column index
    if all(is_int_str(n) for n in nodes):
        idxs = [int(str(n)) for n in nodes]
        if min(idxs) >= 0 and max(idxs) < len(cols):
            return {n: cols[int(str(n))] for n in nodes}

    # 3) remove spaces
    normalized_cols = {re.sub(r"\s+", "", c): c for c in cols}
    mapping: Dict[Any, str] = {}
    for n in nodes:
        key = re.sub(r"\s+", "", str(n))
        if key not in normalized_cols:
            mapping = {}
            break
        mapping[n] = normalized_cols[key]
    if mapping:
        return mapping

    missing = [str(n) for n in nodes if str(n) not in df.columns]
    raise ValueError(
        "Graph nodes do not match dataset columns, and auto-mapping failed.\n"
        f"- Example missing nodes: {missing[:10]}\n"
        f"- Dataset columns (first 20): {cols[:20]}\n"
        "Fix options:\n"
        "1) Rename GEXF nodes to match CSV column names, or\n"
        "2) Add a manual node->column mapping, or\n"
        "3) Remove index-like nodes from graph (DROP_INDEXLIKE_FROM_GRAPH=True)."
    )


def get_edge_weight_optional(attrs: Dict[str, Any]) -> float:
    if not USE_EDGE_WEIGHT:
        return 1.0

    w = attrs.get("weight", None)
    if w is None:
        w = attrs.get("value", None)
    if w is None:
        if STRICT_WEIGHT:
            raise KeyError("missing weight/value")
        return None
    return float(w)


def make_unique(name: str, existing: set) -> str:
    if name not in existing:
        return name
    i = 1
    while f"{name}__dup{i}" in existing:
        i += 1
    return f"{name}__dup{i}"


def iter_edges_with_attrs(G) -> List[Tuple[Any, Any, Dict[str, Any]]]:
    if isinstance(G, nx.MultiDiGraph):
        out = []
        for u, v, k, attrs in G.edges(keys=True, data=True):
            attrs2 = dict(attrs)
            attrs2["_multikey"] = str(k)
            out.append((u, v, attrs2))
        return out
    return list(G.edges(data=True))


def make_shared_split_indices(
    n_rows: int,
    train_ratio: float,
    val_ratio: float,
    test_ratio: float,
    random_state: int,
    shuffle: bool,
) -> Tuple[List[int], List[int], List[int]]:
    s = float(train_ratio) + float(val_ratio) + float(test_ratio)
    if abs(s - 1.0) > 1e-9:
        raise ValueError(f"train+val+test must sum to 1. got {s}")
    if n_rows <= 2:
        raise ValueError(f"Need at least 3 rows to split. n_rows={n_rows}")

    idx = list(range(n_rows))
    if shuffle:
        idx = pd.Series(idx).sample(frac=1.0, random_state=random_state).tolist()

    n_train = int(n_rows * train_ratio)
    n_val = int(n_rows * val_ratio)

    if n_train <= 0:
        n_train = 1
    if n_val <= 0:
        n_val = 1

    if n_train + n_val >= n_rows:
        if n_train >= 2:
            n_train -= 1
        elif n_val >= 2:
            n_val -= 1
        else:
            raise ValueError(f"n_rows={n_rows} too small for non-empty train/val/test splits.")

    train_idx = idx[:n_train]
    val_idx = idx[n_train : n_train + n_val]
    test_idx = idx[n_train + n_val :]
    return train_idx, val_idx, test_idx


def split_and_save_by_indices(
    df: pd.DataFrame,
    out_dir: str,
    base_filename_no_ext: str,
    train_idx: Sequence[int],
    val_idx: Sequence[int],
    test_idx: Sequence[int],
) -> None:
    os.makedirs(out_dir, exist_ok=True)

    df_train = df.iloc[list(train_idx)].copy()
    df_val = df.iloc[list(val_idx)].copy()
    df_test = df.iloc[list(test_idx)].copy()

    df_train.to_csv(os.path.join(out_dir, f"{base_filename_no_ext}_train.csv"), index=False)
    df_val.to_csv(os.path.join(out_dir, f"{base_filename_no_ext}_val.csv"), index=False)
    df_test.to_csv(os.path.join(out_dir, f"{base_filename_no_ext}_test.csv"), index=False)

    print(f"[SPLIT] {base_filename_no_ext}: train={len(df_train)}, val={len(df_val)}, test={len(df_test)}")


def add_edge_features(
    df_base: pd.DataFrame,
    alg: str,
    gexf_path: str,
    out_dir: str,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    df = df_base.copy()
    original_cols = list(df.columns)

    G = ensure_digraph(nx.read_gexf(gexf_path))
    if DROP_INDEXLIKE_FROM_GRAPH:
        G = clean_graph_nodes(G, alg)

    node_to_col = build_node_to_col_mapping(df, G)

    existing_cols = set(df.columns)
    edges_meta: List[Dict[str, Any]] = []
    added_cols: List[str] = []
    skipped_no_weight = 0

    for u, v, attrs in iter_edges_with_attrs(G):
        w = get_edge_weight_optional(attrs)
        if w is None:
            skipped_no_weight += 1
            continue

        u_col = node_to_col[u]
        v_col = node_to_col[v]

        base_name = f"{COL_PREFIX}_{alg}__{str(u)}__{str(v)}"
        if "_multikey" in attrs:
            base_name += f"__k{attrs['_multikey']}"

        new_col = make_unique(base_name, existing_cols)

        # ✅ edge feature
        df[new_col] = w * df[u_col] * df[v_col]

        existing_cols.add(new_col)
        added_cols.append(new_col)

        edges_meta.append(
            {
                "alg": alg,
                "u": str(u),
                "v": str(v),
                "u_col": u_col,
                "v_col": v_col,
                "weight": (w if USE_EDGE_WEIGHT else ""),
                "new_col": new_col,
            }
        )

    os.makedirs(out_dir, exist_ok=True)
    out_csv = os.path.join(out_dir, f"{DATA_PREFIX}_{alg}.csv")
    df.to_csv(out_csv, index=False)

    meta = {
        "alg": alg,
        "data_path": DATA_PATH,
        "gexf_path": gexf_path,
        "output_csv": out_csv,
        "n_rows": int(df.shape[0]),
        "n_original_cols": int(len(original_cols)),
        "n_added_edge_features": int(len(added_cols)),
        "skipped_edges_no_weight": int(skipped_no_weight),
        "added_feature_cols": added_cols,
        "node_to_col_mapping_preview": {str(k): v for k, v in list(node_to_col.items())[:30]},
        "edges": edges_meta,
        "use_edge_weight": bool(USE_EDGE_WEIGHT),
    }

    if WRITE_METADATA:
        out_meta = os.path.join(out_dir, f"metadata_{alg}.json")
        with open(out_meta, "w", encoding="utf-8") as f:
            json.dump(meta, f, ensure_ascii=False, indent=2)

    print(f"[DONE:{alg}] added={len(added_cols)} -> {out_csv}")
    return df, meta


def save_feature_list_csv(
    out_base: str,
    df_base: pd.DataFrame,
    meta: Dict[str, Any],
    filename: str = "feature_list.csv",
    drop_targets: bool = True,
) -> None:
    rows: List[Dict[str, Any]] = []

    target_col = None
    if drop_targets:
        try:
            target_col = detect_target_col(df_base)
        except Exception:
            target_col = None

    # original
    for c in df_base.columns:
        if target_col is not None and str(c) == str(target_col):
            continue
        if is_indexlike_col_name(c):
            continue
        rows.append(
            {
                "GROUP": "original",
                "FEATURE_TYPE": "original",
                "FEATURE_NAME": str(c),
                "ALG": "original",
                "U": "",
                "V": "",
                "U_COL": "",
                "V_COL": "",
                "WEIGHT": "",
                "EXPR": "",
            }
        )

    # edges
    for e in meta.get("edges", []):
        new_col = str(e.get("new_col", ""))
        if not new_col:
            continue

        u_col = str(e.get("u_col", ""))
        v_col = str(e.get("v_col", ""))

        if u_col and v_col:
            expr = f"{u_col} * {v_col}" if not USE_EDGE_WEIGHT else f"{e.get('weight', '')} * {u_col} * {v_col}"
        else:
            expr = ""

        rows.append(
            {
                "GROUP": ALG,
                "FEATURE_TYPE": "edge",
                "FEATURE_NAME": new_col,
                "ALG": ALG,
                "U": str(e.get("u", "")),
                "V": str(e.get("v", "")),
                "U_COL": u_col,
                "V_COL": v_col,
                "WEIGHT": (e.get("weight", "") if USE_EDGE_WEIGHT else ""),
                "EXPR": expr,
            }
        )

    df_out = pd.DataFrame(rows)
    cols = ["GROUP", "FEATURE_TYPE", "FEATURE_NAME", "ALG", "U", "V", "U_COL", "V_COL", "WEIGHT", "EXPR"]
    df_out = df_out[cols]

    os.makedirs(out_base, exist_ok=True)
    out_path = os.path.join(out_base, filename)
    df_out.to_csv(out_path, index=False)
    print(f"[SAVED] feature list -> {out_path} | rows={len(df_out)}")


def main():
    # fail-fast
    if not os.path.exists(DATA_PATH):
        raise FileNotFoundError(f"[FATAL] DATA_PATH not found: {os.path.abspath(DATA_PATH)}")
    if not os.path.exists(GEXF_PATH):
        raise FileNotFoundError(f"[FATAL] GEXF_PATH not found: {os.path.abspath(GEXF_PATH)}")

    out_dir = os.path.join(OUT_BASE, ALG)
    os.makedirs(out_dir, exist_ok=True)

    df_base = pd.read_csv(DATA_PATH, low_memory=False)
    if DROP_INDEXLIKE_FROM_BASE:
        df_base = drop_indexlike_columns(df_base)

    # shared split
    train_idx, val_idx, test_idx = make_shared_split_indices(
        n_rows=len(df_base),
        train_ratio=TRAIN_RATIO,
        val_ratio=VAL_RATIO,
        test_ratio=TEST_RATIO,
        random_state=RANDOM_STATE,
        shuffle=SHUFFLE_BEFORE_SPLIT,
    )

    if SAVE_SPLIT_INDICES:
        split_info_path = os.path.join(OUT_BASE, "split_indices.json")
        with open(split_info_path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "data_path": DATA_PATH,
                    "gexf_path": GEXF_PATH,
                    "alg": ALG,
                    "train_ratio": TRAIN_RATIO,
                    "val_ratio": VAL_RATIO,
                    "test_ratio": TEST_RATIO,
                    "random_state": RANDOM_STATE,
                    "shuffle": SHUFFLE_BEFORE_SPLIT,
                    "n_rows": int(len(df_base)),
                    "train_idx": list(map(int, train_idx)),
                    "val_idx": list(map(int, val_idx)),
                    "test_idx": list(map(int, test_idx)),
                },
                f,
                ensure_ascii=False,
                indent=2,
            )
        print(f"[SPLIT] saved indices -> {split_info_path}")

    # add edge features
    df_out, meta = add_edge_features(
        df_base=df_base,
        alg=ALG,
        gexf_path=GEXF_PATH,
        out_dir=out_dir,
    )

    # split & save (with edge features)
    split_and_save_by_indices(
        df=df_out,
        out_dir=out_dir,
        base_filename_no_ext=f"{DATA_PREFIX}_{ALG}",
        train_idx=train_idx,
        val_idx=val_idx,
        test_idx=test_idx,
    )

    # feature list
    if WRITE_FEATURE_LIST:
        save_feature_list_csv(
            out_base=OUT_BASE,
            df_base=df_base,
            meta=meta,
            filename=FEATURE_LIST_FILENAME,
            drop_targets=True,
        )

    print(f"[ALL DONE] OUT_DIR = {os.path.abspath(out_dir)}")


if __name__ == "__main__":
    main()


[SPLIT] saved indices -> ./add_feature_dataset\split_indices.json
[DONE:GES] added=35 -> ./add_feature_dataset\GES\data_with_features_GES.csv
[SPLIT] data_with_features_GES: train=12516, val=1788, test=3577
[SAVED] feature list -> ./add_feature_dataset\feature_list.csv | rows=48
[ALL DONE] OUT_DIR = d:\University\3-2.5\PADA_Lab\Bank_Failure_Prediction_3\9_best_model_no_weight\add_feature_dataset\GES
